# Pass 3 — Verify Objectives Against Source Text

**Input:** Pass 2 consolidated objectives + original source columns.  
**Task:** For each consolidated objective, the LLM checks whether it can be found/traced in ANY of the original source columns.  

**Output:** Each objective gets a verification status:  
- `VERIFIED` — objective text found in at least one source column  
- `FLAGGED` — cannot be traced back → flagged for manual review  

The LLM also re-checks the classification (financial/sustainable) for correctness.

In [3]:
import pandas as pd
from tqdm import tqdm
import time, os, json, anthropic
from pathlib import Path

with open("Claude_API.txt", "r") as file:
    api_key = file.read().strip()
os.environ['ANTHROPIC_API_KEY'] = api_key

config = {}
with open("File_Directory.txt", "r") as file:
    for line in file:
        if ":" in line:
            key, value = line.split(":", 1)
            config[key.strip()] = value.strip()

INPUT_FILE = Path(config["Input"])
OUTPUT_DIR = Path(config["Output"])
MODEL = "claude-sonnet-4-6"  # UPDATE as needed

OBJECTIVE_COLUMNS = [
    'PRIIPS KID Objective',
    'KIID Objective/Investment Policy',
    'Prospectus Objective',
    'Investment Strategy - English',
    'PRIIPS KID Objective - Danish',
    'PRIIPS KID Objective - Dutch',
    'PRIIPS KID Objective - Finnish',
    'PRIIPS KID Objective - French',
    'PRIIPS KID Objective - German',
    'PRIIPS KID Objective - Italian',
    'PRIIPS KID Objective - Norwegian',
    'PRIIPS KID Objective - Portuguese',
    'PRIIPS KID Objective - Spanish',
    'PRIIPS KID Objective - Swedish',
    'KIID Objective/Investment Policy - German',
    'KIID Objective/Investment Policy - French',
    'KIID Objective/Investment Policy - Italian',
    'KIID Objective/Investment Policy - Spanish',
    'KIID Objective/Investment Policy - Norwegian',
    'KIID Objective/Investment Policy - Swedish',
    'KIID Objective/Investment Policy - Finnish',
    'KIID Objective/Investment Policy - Portuguese',
    'KIID Objective/Investment Policy - Danish',
    'Investment Strategy - Danish',
    'Investment Strategy - Finnish',
    'Investment Strategy - French',
    'Investment Strategy - German',
    'Investment Strategy - Italian',
    'Investment Strategy - Norwegian',
    'Investment Strategy - Portuguese',
    'Investment Strategy - Spanish',
    'Investment Strategy - Swedish',
    'Strategy Description'
]

In [4]:
# === LOAD INPUTS ===

# Pass 2 raw output (with JSON)
# UPDATE this path to your actual Pass 2 raw output file
PASS2_FILE = os.path.join(OUTPUT_DIR, "Pass2_Raw_100_funds_20260518_1335.xlsx")  # UPDATE

p2_df = pd.read_excel(PASS2_FILE)
p2_df['pass2_raw'] = p2_df['pass2_raw'].apply(json.loads)
print(f"Loaded {len(p2_df)} funds from Pass 2")

# Original source data (for verification)
df_source = pd.read_excel(INPUT_FILE)
print(f"Loaded {len(df_source)} funds from source data")

Loaded 100 funds from Pass 2
Loaded 5680 funds from source data


In [5]:
PASS3_SYSTEM_PROMPT = """You are verifying extracted fund objectives against the original regulatory source text.

You will receive:
1. A list of consolidated objectives (in English) from Pass 2
2. The original source text from ALL available columns (in various languages)

YOUR TASK:
For EACH objective, determine whether it can be traced back to text in ANY of the source columns.

VERIFICATION RULES:
- An objective is VERIFIED if you can find corresponding text in at least one source column.
  The match can be in any language — the objective may be an English translation of French/German/etc. source text.
- An objective is FLAGGED if you cannot find any corresponding text in any column.
  This means it may have been hallucinated or incorrectly inferred.

CLASSIFICATION CHECK:
- Also verify whether each objective is correctly classified as "financial" or "sustainable".
- If the classification is wrong, provide the correct one.

OUTPUT FORMAT:
{
  "verified_objectives": [
    {
      "objective_number": 1,
      "objective_text_english": "the objective text from Pass 2",
      "verification_status": "VERIFIED" or "FLAGGED",
      "verified_in_column": "column name where found" or null,
      "source_quote": "brief quote or paraphrase from source showing the match" or null,
      "objective_type": "financial" or "sustainable",
      "type_changed": false,
      "verification_notes": "brief explanation"
    }
  ],
  "overall_confidence": "high" or "medium" or "low",
  "verification_summary": "brief summary of verification results"
}
"""

In [6]:
def get_source_columns_text(fund_id, df_source, objective_columns):
    """Get all non-empty source column text for a fund."""
    fund_row = df_source[df_source['FundId'] == fund_id]
    if fund_row.empty:
        return {}
    row = fund_row.iloc[0]
    columns = {}
    for col in objective_columns:
        if col in row.index:
            value = row[col]
            if pd.notna(value) and str(value).strip() not in ['-', 'Not available', '']:
                columns[col] = str(value)
    return columns


def pass3_verify(fund_name, fund_id, pass2_objectives, source_columns):
    """Verify each objective against the original source text."""
    if not pass2_objectives:
        return {
            "verified_objectives": [],
            "overall_confidence": "none",
            "verification_summary": "No objectives to verify"
        }

    source_text = "\n\n".join(
        [f"=== Column: {col} ===\n{val}" for col, val in source_columns.items()]
    )

    objectives_text = json.dumps(pass2_objectives, indent=2)

    user_prompt = f"""Fund ID: {fund_id}
Fund Name: {fund_name}

CONSOLIDATED OBJECTIVES FROM PASS 2:
{objectives_text}

ORIGINAL SOURCE TEXT (all available columns):
{source_text}"""

    messages = [{"role": "user", "content": user_prompt}]

    try:
        client = anthropic.Anthropic()
        response = client.messages.create(
            model=MODEL,
            max_tokens=2000,
            temperature=0,
            system=PASS3_SYSTEM_PROMPT,
            messages=messages
        )
        print(f"   [{fund_name}] tokens — in: {response.usage.input_tokens}, out: {response.usage.output_tokens}")

        text = response.content[0].text
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            if "```json" in text:
                return json.loads(text.split("```json")[1].split("```")[0].strip())
            elif "```" in text:
                return json.loads(text.split("```")[1].split("```")[0].strip())
            return {"_error": f"JSON parse error: {text[:300]}"}

    except Exception as e:
        print(f"   Error for {fund_name}: {e}")
        return {"_error": str(e)}

In [7]:
# === RUN PASS 3 ===
pass3_results = []

for idx in tqdm(range(len(p2_df)), desc="Pass 3 — Verify"):
    row = p2_df.iloc[idx]
    fund_id = row['FundId']
    fund_name = row['Fund_Name']
    p2_data = row['pass2_raw']

    # Skip errors
    if '_error' in p2_data:
        pass3_results.append({
            'FundId': fund_id,
            'Fund_Name': fund_name,
            'pass3_raw': {'_error': f"Skipped — Pass 2 error: {p2_data['_error']}"}
        })
        continue

    objectives = p2_data.get('consolidated_objectives', [])
    if not objectives:
        pass3_results.append({
            'FundId': fund_id,
            'Fund_Name': fund_name,
            'pass3_raw': {
                'verified_objectives': [],
                'overall_confidence': 'none',
                'verification_summary': 'No objectives from Pass 2'
            }
        })
        continue

    source_columns = get_source_columns_text(fund_id, df_source, OBJECTIVE_COLUMNS)
    result = pass3_verify(fund_name, fund_id, objectives, source_columns)

    pass3_results.append({
        'FundId': fund_id,
        'Fund_Name': fund_name,
        'pass3_raw': result
    })

    if idx > 0 and idx % 50 == 0:
        time.sleep(0.5)

pass3_df = pd.DataFrame(pass3_results)
print(f"\nPass 3 complete: {len(pass3_df)} funds processed")

Pass 3 — Verify:   1%|          | 1/100 [00:10<17:55, 10.86s/it]

   [MS INVF Global Brands Eq Inc Z] tokens — in: 9807, out: 480


Pass 3 — Verify:   2%|▏         | 2/100 [00:23<19:52, 12.17s/it]

   [DWS Global Value LD] tokens — in: 7407, out: 931


Pass 3 — Verify:   3%|▎         | 3/100 [00:31<16:33, 10.24s/it]

...

   [JB Edelweiss Swiss Equity SK Acc CHF] tokens — in: 12422, out: 475


Pass 3 — Verify:  98%|█████████▊| 98/100 [15:31<00:19,  9.63s/it]

   [WealthInv Qblue Bal GlbAkt AnsTran I] tokens — in: 2395, out: 478


Pass 3 — Verify: 100%|██████████| 100/100 [15:37<00:00,  9.38s/it]

   [Ethos Aktiefond A Utdelande (SEK)] tokens — in: 2467, out: 363

Pass 3 complete: 100 funds processed


In [8]:
# === FLATTEN INTO FINAL OUTPUT ===
final_rows = []

for _, row in pass3_df.iterrows():
    raw = row['pass3_raw']
    base = {
        'FundId': row['FundId'],
        'Fund_Name': row['Fund_Name']
    }

    if '_error' in raw:
        base['Number_of_Objectives'] = 0
        base['Overall_Confidence'] = 'error'
        base['Verification_Summary'] = raw['_error']
        base['Has_Flagged'] = False
        final_rows.append(base)
        continue

    objs = raw.get('verified_objectives', [])
    base['Number_of_Objectives'] = len(objs)
    base['Overall_Confidence'] = raw.get('overall_confidence', '')
    base['Verification_Summary'] = raw.get('verification_summary', '')

    flagged = any(o.get('verification_status') == 'FLAGGED' for o in objs)
    base['Has_Flagged'] = flagged

    for i in range(5):
        if i < len(objs):
            o = objs[i]
            base[f'Objective_{i+1}'] = o.get('objective_text_english', '')
            base[f'Objective_{i+1}_Type'] = o.get('objective_type', '')
            base[f'Objective_{i+1}_Status'] = o.get('verification_status', '')
            base[f'Objective_{i+1}_Verified_In'] = o.get('verified_in_column', '')
            base[f'Objective_{i+1}_Type_Changed'] = o.get('type_changed', False)
            base[f'Objective_{i+1}_Notes'] = o.get('verification_notes', '')
        else:
            base[f'Objective_{i+1}'] = None
            base[f'Objective_{i+1}_Type'] = None
            base[f'Objective_{i+1}_Status'] = None
            base[f'Objective_{i+1}_Verified_In'] = None
            base[f'Objective_{i+1}_Type_Changed'] = None
            base[f'Objective_{i+1}_Notes'] = None

    final_rows.append(base)

final_df = pd.DataFrame(final_rows)

print("=" * 80)
print("FINAL VERIFICATION SUMMARY")
print("=" * 80)
total = len(final_df)
with_obj = (final_df['Number_of_Objectives'] > 0).sum()
flagged_funds = final_df['Has_Flagged'].sum()

print(f"  Funds processed: {total}")
print(f"  Funds with objectives: {with_obj} ({with_obj/total*100:.1f}%)")
print(f"  Funds with FLAGGED objectives: {flagged_funds} ({flagged_funds/total*100:.1f}%)")

# Count individual objective statuses
all_statuses = []
for col in [f'Objective_{i}_Status' for i in range(1, 6)]:
    all_statuses.extend(final_df[col].dropna().tolist())

if all_statuses:
    from collections import Counter
    status_counts = Counter(all_statuses)
    print(f"\n  Objective-level verification:")
    for s, c in status_counts.items():
        print(f"    {s}: {c} ({c/len(all_statuses)*100:.1f}%)")

# Count type changes
type_changes = []
for col in [f'Objective_{i}_Type_Changed' for i in range(1, 6)]:
    type_changes.extend([v for v in final_df[col].dropna() if v == True])
print(f"\n  Classification changes: {len(type_changes)}")

print(f"\nConfidence distribution:")
print(final_df['Overall_Confidence'].value_counts())

FINAL VERIFICATION SUMMARY
  Funds processed: 100
  Funds with objectives: 92 (92.0%)
  Funds with FLAGGED objectives: 0 (0.0%)

  Objective-level verification:
    VERIFIED: 161 (100.0%)

  Classification changes: 0

Confidence distribution:
Overall_Confidence
high      91
error      6
none       2
medium     1
Name: count, dtype: int64


In [9]:
# === SHOW FLAGGED OBJECTIVES FOR MANUAL REVIEW ===
flagged_df = final_df[final_df['Has_Flagged'] == True]

if len(flagged_df) > 0:
    print(f"\n{'='*80}")
    print(f"FLAGGED FOR MANUAL REVIEW: {len(flagged_df)} funds")
    print(f"{'='*80}")
    for _, row in flagged_df.iterrows():
        print(f"\n  Fund: {row['Fund_Name']} ({row['FundId']})")
        for i in range(1, 6):
            status = row.get(f'Objective_{i}_Status')
            if status == 'FLAGGED':
                print(f"    FLAGGED Objective {i}: {row[f'Objective_{i}']}")
                print(f"      Type: {row[f'Objective_{i}_Type']}")
                print(f"      Notes: {row[f'Objective_{i}_Notes']}")
else:
    print("\nNo flagged objectives — all verified successfully.")


No flagged objectives — all verified successfully.


In [10]:
# === SAVE FINAL OUTPUT ===
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M")

# Final verified results
final_filename = f'FINAL_Verified_{len(final_df)}_funds_{timestamp}.xlsx'
final_path = os.path.join(OUTPUT_DIR, final_filename)
final_df.to_excel(final_path, index=False, engine='openpyxl')

# Flagged-only file for manual review
if len(flagged_df) > 0:
    flagged_filename = f'FLAGGED_ManualReview_{len(flagged_df)}_funds_{timestamp}.xlsx'
    flagged_path = os.path.join(OUTPUT_DIR, flagged_filename)
    flagged_df.to_excel(flagged_path, index=False, engine='openpyxl')
    print(f"Saved flagged:  {flagged_filename}")

print(f"Saved final:    {final_filename}")
print(f"\nDone. Three-pass extraction complete.")

Saved final:    FINAL_Verified_100_funds_20260518_1500.xlsx

Done. Three-pass extraction complete.
